In [22]:
# ============================================================
# OLIST CUSTOMER PURCHASE ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import csv

# ============================================================
# 1. LOAD DATA
# ============================================================

customers = pd.read_csv('/content/sample_data/customers.csv')
orders = pd.read_csv('/content/sample_data/orders.csv', quoting=csv.QUOTE_NONE)
order_items = pd.read_csv('/content/sample_data/order_items.csv', quoting=csv.QUOTE_NONE)
payments = pd.read_csv('/content/sample_data/order_payments.csv', quoting=csv.QUOTE_NONE)

print("Data Loaded Successfully")
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

customers.columns = customers.columns.str.replace('"', '', regex=False).str.strip()
orders.columns = orders.columns.str.replace('"', '', regex=False).str.strip()
order_items.columns = order_items.columns.str.replace('"', '', regex=False).str.strip()
payments.columns = payments.columns.str.replace('"', '', regex=False).str.strip()

print("\nColumn names cleaned")


# ============================================================
# 3. CHECK MISSING VALUES
# ============================================================

print("\nCUSTOMERS MISSING VALUES")
print(customers.isnull().sum())

print("\nORDERS MISSING VALUES")
print(orders.isnull().sum())

print("\nORDER ITEMS MISSING VALUES")
print(order_items.isnull().sum())

print("\nPAYMENTS MISSING VALUES")
print(payments.isnull().sum())


# ============================================================
# 4. CHECK ORDER STATUS
# ============================================================

print("\nORDER STATUS")
print(orders['order_status'].value_counts(dropna=False))


# ============================================================
# 5. REMOVE INVALID ORDERS
# ============================================================

# Orders without customer/order status/purchase date
orders = orders.dropna(
    subset=['order_id', 'customer_id',
            'order_status', 'order_purchase_timestamp']
)

print("\nOrders after removing invalid records:", orders.shape)


# ============================================================
# 6. CONVERT DATE COLUMNS
# ============================================================

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date'],
    errors='coerce'
)


# ============================================================
# 7. CHECK ORDER ITEMS WITH MISSING VALUES
# ============================================================

print("\nORDER ITEMS WITH MISSING VALUES")
print(
    order_items[
        order_items[['seller_id', 'shipping_limit_date',
                     'price', 'freight_value']].isnull().any(axis=1)
    ]
)


# ============================================================
# 8. REMOVE ORDER ITEMS WITHOUT PRICE
# ============================================================

# Price is required for purchase/revenue analysis
order_items = order_items.dropna(
    subset=['order_id', 'product_id', 'price', 'freight_value']
)

print("\nOrder Items after cleaning:", order_items.shape)


# ============================================================
# 9. REMOVE CANCELLED / UNAVAILABLE ORDERS
# ============================================================

# For purchase analysis, use completed orders.
valid_status = [
    'delivered',
    'shipped',
    'invoiced',
    'approved',
    'processing'
]

orders_analysis = orders[
    orders['order_status'].isin(valid_status)
].copy()

print("\nOrders used for purchase analysis:",
      orders_analysis.shape)


# ============================================================
# 10. JOIN CUSTOMERS WITH ORDERS
# ============================================================

customer_orders = orders_analysis.merge(
    customers,
    on='customer_id',
    how='inner'
)

print("\nCustomer + Orders:")
print(customer_orders.shape)


# ============================================================
# 11. JOIN ORDER ITEMS
# ============================================================

customer_order_items = customer_orders.merge(
    order_items,
    on='order_id',
    how='inner'
)

print("\nCustomer + Orders + Order Items:")
print(customer_order_items.shape)


# ============================================================
# 12. CALCULATE ITEM REVENUE
# ============================================================

customer_order_items['item_revenue'] = (
    customer_order_items['price'] +
    customer_order_items['freight_value']
)


# ============================================================
# 13. CUSTOMER LEVEL PURCHASE ANALYSIS
# ============================================================

customer_analysis = customer_order_items.groupby(
    'customer_unique_id'
).agg(
    total_orders=('order_id', 'nunique'),
    total_items=('order_item_id', 'count'),
    total_purchase=('item_revenue', 'sum'),
    average_order_value=('item_revenue', 'sum')
).reset_index()


# ============================================================
# 14. CORRECT AOV
# ============================================================

customer_analysis['average_order_value'] = (
    customer_analysis['total_purchase'] /
    customer_analysis['total_orders']
)


# ============================================================
# 15. PURCHASE FREQUENCY
# ============================================================

customer_analysis['purchase_frequency'] = (
    customer_analysis['total_orders'] /
    customer_analysis['total_items']
)


# ============================================================
# 16. CUSTOMER TYPE
# ============================================================

customer_analysis['customer_type'] = np.where(
    customer_analysis['total_orders'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)


# ============================================================
# 17. LAST PURCHASE DATE
# ============================================================

last_purchase = customer_order_items.groupby(
    'customer_unique_id'
)['order_purchase_timestamp'].max().reset_index()

last_purchase.rename(
    columns={
        'order_purchase_timestamp': 'last_purchase_date'
    },
    inplace=True
)

customer_analysis = customer_analysis.merge(
    last_purchase,
    on='customer_unique_id',
    how='left'
)


# ============================================================
# 18. FIRST PURCHASE DATE
# ============================================================

first_purchase = customer_order_items.groupby(
    'customer_unique_id'
)['order_purchase_timestamp'].min().reset_index()

first_purchase.rename(
    columns={
        'order_purchase_timestamp': 'first_purchase_date'
    },
    inplace=True
)

customer_analysis = customer_analysis.merge(
    first_purchase,
    on='customer_unique_id',
    how='left'
)


# ============================================================
# 19. CUSTOMER LIFESPAN
# ============================================================

customer_analysis['customer_lifespan_days'] = (
    customer_analysis['last_purchase_date'] -
    customer_analysis['first_purchase_date']
).dt.days


# ============================================================
# 20. CUSTOMER SUMMARY
# ============================================================

print("\n================ CUSTOMER ANALYSIS ================")

print("\nTotal Customers:",
      customer_analysis['customer_unique_id'].nunique())

print("Total Orders:",
      customer_analysis['total_orders'].sum())

print("Total Purchase:",
      customer_analysis['total_purchase'].sum())

print("Average Order Value:",
      customer_analysis['average_order_value'].mean())


# ============================================================
# 21. REPEAT VS ONE-TIME CUSTOMERS
# ============================================================

customer_type_summary = (
    customer_analysis
    .groupby('customer_type')
    .agg(
        customers=('customer_unique_id', 'count'),
        total_purchase=('total_purchase', 'sum'),
        average_purchase=('total_purchase', 'mean'),
        average_orders=('total_orders', 'mean')
    )
    .reset_index()
)

print("\nCUSTOMER TYPE SUMMARY")
print(customer_type_summary)


# ============================================================
# 22. TOP 10 CUSTOMERS BY PURCHASE
# ============================================================

top_customers = customer_analysis.sort_values(
    'total_purchase',
    ascending=False
).head(10)

print("\nTOP 10 CUSTOMERS")
print(top_customers)


# ============================================================
# 23. MONTHLY PURCHASE ANALYSIS
# ============================================================

customer_order_items['purchase_month'] = (
    customer_order_items['order_purchase_timestamp']
    .dt.to_period('M')
    .astype(str)
)

monthly_purchase = (
    customer_order_items
    .groupby('purchase_month')
    .agg(
        customers=('customer_unique_id', 'nunique'),
        orders=('order_id', 'nunique'),
        purchase_amount=('item_revenue', 'sum')
    )
    .reset_index()
)

monthly_purchase['AOV'] = (
    monthly_purchase['purchase_amount'] /
    monthly_purchase['orders']
)

print("\nMONTHLY PURCHASE")
print(monthly_purchase)


# ============================================================
# 24. CUSTOMER STATE ANALYSIS
# ============================================================

state_analysis = (
    customer_order_items
    .groupby('customer_state')
    .agg(
        customers=('customer_unique_id', 'nunique'),
        orders=('order_id', 'nunique'),
        purchase_amount=('item_revenue', 'sum')
    )
    .reset_index()
)

state_analysis['AOV'] = (
    state_analysis['purchase_amount'] /
    state_analysis['orders']
)

state_analysis = state_analysis.sort_values(
    'purchase_amount',
    ascending=False
)

print("\nSTATE ANALYSIS")
print(state_analysis)


# ============================================================
# 25. PAYMENT ANALYSIS
# ============================================================

payment_analysis = payments.merge(
    orders_analysis[['order_id', 'customer_id']],
    on='order_id',
    how='inner'
)

payment_analysis = payment_analysis.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
)

payment_summary = (
    payment_analysis
    .groupby('payment_type')
    .agg(
        customers=('customer_unique_id', 'nunique'),
        orders=('order_id', 'nunique'),
        payment_amount=('payment_value', 'sum'),
        average_payment=('payment_value', 'mean')
    )
    .reset_index()
)

print("\nPAYMENT ANALYSIS")
print(payment_summary)


# ============================================================
# 26. CUSTOMER PURCHASE SEGMENTS
# ============================================================

customer_analysis['purchase_segment'] = pd.cut(
    customer_analysis['total_purchase'],
    bins=[0, 100, 500, 1000, 5000, np.inf],
    labels=[
        'Low',
        'Medium',
        'High',
        'Very High',
        'VIP'
    ]
)

segment_analysis = (
    customer_analysis
    .groupby('purchase_segment', observed=False)
    .agg(
        customers=('customer_unique_id', 'count'),
        total_purchase=('total_purchase', 'sum'),
        average_purchase=('total_purchase', 'mean')
    )
    .reset_index()
)

print("\nPURCHASE SEGMENTS")
print(segment_analysis)


# ============================================================
# 27. FINAL DATASET FOR POWER BI / EXCEL
# ============================================================

final_customer_purchase = customer_analysis[
    [
        'customer_unique_id',
        'total_orders',
        'total_items',
        'total_purchase',
        'average_order_value',
        'purchase_frequency',
        'customer_type',
        'first_purchase_date',
        'last_purchase_date',
        'customer_lifespan_days',
        'purchase_segment'
    ]
].copy()


# ============================================================
# 28. SAVE ANALYSIS FILES
# ============================================================

customer_analysis.to_csv(
    '/content/customer_purchase_analysis.csv',
    index=False
)

monthly_purchase.to_csv(
    '/content/monthly_purchase_analysis.csv',
    index=False
)

state_analysis.to_csv(
    '/content/state_purchase_analysis.csv',
    index=False
)

payment_summary.to_csv(
    '/content/payment_analysis.csv',
    index=False
)

segment_analysis.to_csv(
    '/content/customer_segment_analysis.csv',
    index=False
)

print("\n================================================")
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("================================================")

print("\nFiles created:")
print("1. customer_purchase_analysis.csv")
print("2. monthly_purchase_analysis.csv")
print("3. state_purchase_analysis.csv")
print("4. payment_analysis.csv")
print("5. customer_segment_analysis.csv")

Data Loaded Successfully
Customers: (99441, 5)
Orders: (17714, 8)
Order Items: (68485, 7)
Payments: (103886, 5)

Column names cleaned

CUSTOMERS MISSING VALUES
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

ORDERS MISSING VALUES
order_id                           0
customer_id                        1
order_status                       1
order_purchase_timestamp           1
order_approved_at                 34
order_delivered_carrier_date     302
order_delivered_customer_date    505
order_estimated_delivery_date      1
dtype: int64

ORDER ITEMS MISSING VALUES
order_id               0
order_item_id          0
product_id             0
seller_id              1
shipping_limit_date    1
price                  1
freight_value          1
dtype: int64

PAYMENTS MISSING VALUES
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
